messages的类型

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek

from dotenv import load_dotenv
load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

def llm_node(state: MessagesState) -> MessagesState:
    messages = state["messages"]
    response = model.invoke(messages)

    return {
        "messages": [response],
    }

builder = StateGraph(state_schema=MessagesState)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

# 使用流式输出
for chunk in graph.stream(
        {
            "messages":[HumanMessage(content="你好!")]
        },
    stream_mode=["values","messages"], # messages只能显示增量，如果不打印values，看不到最后说了什么
):
    print(chunk)


# 看下面内容，第一个AIMessage是空的， 之后才回复， 然后内容一个个蹦出来，  大模型底层本质是流式输出
# 如果直接invoke，会看不出来，因为只打印了最终的结果，上面回复空就会消耗token，空信息也有很多response_metadata
# 只需要注意每次增量一个个出来

('values', {'messages': [HumanMessage(content='你好!', additional_kwargs={}, response_metadata={}, id='7ee52e9d-8d5b-46dd-8663-137bb2e69ec9')]})
('messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--01a0b38e-6e6c-7bf3-b9eb-699b6d4f1fa8', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'langgraph_step': 1, 'langgraph_node': 'llm_node', 'langgraph_triggers': ('branch:to:llm_node',), 'langgraph_path': ('__pregel_pull', 'llm_node'), 'langgraph_checkpoint_ns': 'llm_node:8a960bb4-0b75-8a14-ec14-0a53a9c1f3c0', 'checkpoint_ns': 'llm_node:8a960bb4-0b75-8a14-ec14-0a53a9c1f3c0', 'ls_provider': 'deepseek', 'ls_model_name': 'deepseek-v4-flash', 'ls_model_type': 'chat', 'ls_temperature': None}))
('messages', (AIMessageChunk(content='你好', additional_kwargs={}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--01a0b38e-6e6c-7bf3-b9eb-699b6d4f1fa8', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {